# Hooks: Intercept and Control Agent Behavior

In this notebook, we'll explore how to use hooks to intercept agent execution at key points and add validation, logging, security controls, or custom logic.

## Setup

Configure the environment:

In [1]:
# Setup for running async code in Jupyter
import nest_asyncio
nest_asyncio.apply()

# Load optional environment variables from a .env file (if present).
# This course uses your Claude subscription, so no ANTHROPIC_API_KEY is needed --
# load_dotenv() is here only for optional keys (e.g. TAVILY_API_KEY), or an
# ANTHROPIC_API_KEY if you choose per-token API billing instead.
from dotenv import load_dotenv
load_dotenv()

print("✓ Notebook environment configured")

✓ Notebook environment configured


## Authentication

This course runs on your **Claude Pro/Max subscription** — no API key or per-token billing required.

**One-time setup:**

1. Install the Claude Code CLI. The Python SDK drives it under the hood (it is *not* bundled with `claude-agent-sdk`):
   ```bash
   # macOS / Linux (native installer)
   curl -fsSL https://claude.ai/install.sh | bash
   # or with npm:  npm install -g @anthropic-ai/claude-code
   ```
2. Log in with your Claude account:
   ```bash
   claude        # opens a browser to log in (use /login if not prompted)
   ```
   Inside `claude`, run `/status` to confirm you're authenticated via your subscription.

> ⚠️ **Precedence gotcha:** if `ANTHROPIC_API_KEY` is set in your environment, the SDK uses it (per-token API billing) and **ignores your subscription**. For subscription use, keep `ANTHROPIC_API_KEY` unset — don't put it in `.env`.

See the [Claude Code quickstart](https://code.claude.com/docs/en/quickstart) for Windows and other install options.

### Optional: use an API key instead

Prefer per-token API billing? You don't need to change any code — just make the key available in the environment and the SDK picks it up automatically:

```bash
# Option A - shell environment variable
export ANTHROPIC_API_KEY="sk-ant-..."
```

```bash
# Option B - add to a .env file in the project (the setup cell above calls load_dotenv())
# ANTHROPIC_API_KEY=sk-ant-...
```

```python
# (Optional) confirm which auth the SDK will use:
# import os
# print("API key billing" if os.environ.get("ANTHROPIC_API_KEY") else "Claude subscription")
```

## Why Hooks?

Hooks let you intercept agent execution at key points to:

- **Block dangerous operations** before they execute (like `rm -rf /` or unauthorized file access)
- **Log and audit** every tool call for compliance, debugging, or analytics
- **Transform inputs and outputs** to sanitize data, inject credentials, or redirect paths
- **Require human approval** for sensitive actions like database writes or API calls
- **Track session lifecycle** to manage state, clean up resources, or send notifications

### Hook Structure

A hook has two parts:
1. **The callback function**: The logic that runs when the hook fires
2. **The hook configuration**: Tells the SDK which event to hook into (like `PreToolUse`) and which tools to match

## Helper Function for Clean Output

First, let's create a helper to print agent messages in a readable format:

In [4]:
import json

def print_message(message):
    """Pretty print agent messages."""
    msg_type = type(message).__name__
    
    if msg_type == "SystemMessage":
        # Skip system messages for cleaner output
        pass
    
    elif msg_type == "AssistantMessage":
        # Print assistant thinking and tool use
        if hasattr(message, 'content'):
            for block in message.content:
                block_type = type(block).__name__
                if block_type == "TextBlock":
                    print(f"\nAssistant: {block.text}")
                elif block_type == "ToolUseBlock":
                    print(f"\nTool Call: {block.name}")
                    if hasattr(block, 'input'):
                        # Show description first if available
                        if 'description' in block.input:
                            print(f"  Description: {block.input['description']}")
                        # Show other arguments (excluding description)
                        args = {k: v for k, v in block.input.items() if k != 'description'}
                        if args:
                            print(f"  Arguments: {json.dumps(args, indent=6)}")
    
    elif msg_type == "UserMessage":
        # Print tool results
        if hasattr(message, 'content'):
            for block in message.content:
                block_type = type(block).__name__
                if block_type == "ToolResultBlock":
                    if block.is_error:
                        print(f"\nTool Error: {block.content}")
                    else:
                        # Show first 500 chars of result
                        content = str(block.content)
                        if len(content) > 500:
                            content = content[:500] + "..."
                        print(f"\nTool Result: {content}")
    
    elif msg_type == "ResultMessage":
        # Only show cost and timing metadata
        if hasattr(message, 'total_cost_usd') and hasattr(message, 'duration_ms'):
            print(f"\nCost: ${message.total_cost_usd:.4f} | Time: {message.duration_ms/1000:.1f}s")

print("Helper function ready!")

Helper function ready!


## Example 1: Blocking Bash Commands

Let's create a hook that blocks `ls` commands to demonstrate how hooks can prevent operations.

**Important:** Hooks require using `ClaudeSDKClient` (streaming mode), not the simple `query()` function.

In [5]:
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, HookMatcher
import json

# Define a hook callback that receives tool call details
async def block_ls_commands(input_data, tool_use_id, context):
    """
    Block all 'ls' commands to demonstrate hook blocking.
    
    Args:
        input_data: Event details (hook_event_name, tool_name, tool_input, etc.)
        tool_use_id: Correlate PreToolUse and PostToolUse events
        context: Reserved for future use in Python
    """
    print("\n" + "=" * 60)
    print(f"HOOK FIRED: {input_data.get('hook_event_name', 'unknown')}")
    print("=" * 60)
    print(f"Tool Name: {input_data.get('tool_name', 'unknown')}")
    print(f"Tool Use ID: {tool_use_id}")
    print(f"Tool Input: {json.dumps(input_data.get('tool_input', {}), indent=2)}")
    print("=" * 60)
    
    # Extract the command from tool input
    command = input_data.get('tool_input', {}).get('command', '')
    
    # Check if it's an ls command
    if command.strip().startswith('ls'):
        print("DECISION: BLOCK THIS COMMAND")
        print("PreToolUse: DENIED")
        print("=" * 60 + "\n")
        
        # Block the command
        return {
            'hookSpecificOutput': {
                'hookEventName': input_data['hook_event_name'],
                'permissionDecision': 'deny',
                'permissionDecisionReason': 'ls commands are blocked by security policy'
            }
        }
    
    print("DECISION: ALLOW THIS COMMAND")
    print("PreToolUse: ALLOWED")
    print("=" * 60 + "\n")
    return {}

# Test the hook
async def test_block_ls():
    print("=" * 60)
    print("Testing ls command blocking")
    print("=" * 60)
    
    # Configure hooks in options
    options = ClaudeAgentOptions(
        hooks={
            # Register the hook for PreToolUse events
            # The matcher filters to only Bash tool calls
            'PreToolUse': [HookMatcher(matcher='Bash', hooks=[block_ls_commands])]
        },
        permission_mode="bypassPermissions"  # Auto-approve for demo
    )
    
    # Use ClaudeSDKClient for hooks to work
    async with ClaudeSDKClient(options) as client:
        # Send message
        async def send_message():
            yield {
                "type": "user",
                "message": {
                    "role": "user",
                    "content": "Run this bash command: ls -la"
                }
            }
        
        await client.query(send_message())
        
        # Receive and display response
        async for message in client.receive_response():
            print_message(message)
            if type(message).__name__ == "ResultMessage":
                break

await test_block_ls()

Testing ls command blocking

Assistant: I'll run that command for you.

Tool Call: Bash
  Description: List files in current directory
  Arguments: {
      "command": "ls -la"
}

HOOK FIRED: PreToolUse
Tool Name: Bash
Tool Use ID: toolu_01Ui5aaobzdV5cedo39rL3UF
Tool Input: {
  "command": "ls -la",
  "description": "List files in current directory"
}
DECISION: BLOCK THIS COMMAND
PreToolUse: DENIED


Tool Error: ls commands are blocked by security policy

Assistant: The `ls -la` command is blocked by this environment's security policy, so I wasn't able to run it.

If you'd like to see the directory contents, I can use an alternative approach instead. For example, I could use a tool-based directory listing or a different command. Just let me know — or tell me what you're actually trying to find and I can help with that directly.

Cost: $0.0561 | Time: 11.0s


### How It Works

1. **Use ClaudeSDKClient** - Hooks require streaming mode with `ClaudeSDKClient`, not the simple `query()` function
2. **Hook callback fires** before the tool executes (PreToolUse)
3. **Inspect tool details** - see the tool name, arguments, and input data
4. **Apply custom logic** - check if command starts with `ls`
5. **Make decision** - return empty `{}` to allow or `hookSpecificOutput` with `permissionDecision: 'deny'` to block
6. **Agent responds** - if blocked, the agent sees the denial and explains to the user

**Key insight:** The `print_message()` helper shows you the agent's full workflow - what it's thinking, which tools it calls, and the results.

## Example 2: Protecting Notebook Files

Block access to Jupyter notebook files to prevent sensitive code/data exposure:

In [ ]:
import json

async def protect_notebook_files(input_data, tool_use_id, context):
    """
    Prevent access to Jupyter notebook files.
    """
    print("\n" + "=" * 60)
    print(f"HOOK FIRED: {input_data.get('hook_event_name', 'unknown')}")
    print("=" * 60)
    print(f"Tool Name: {input_data.get('tool_name', 'unknown')}")
    print(f"Tool Use ID: {tool_use_id}")
    print(f"Tool Input: {json.dumps(input_data.get('tool_input', {}), indent=2)}")
    print("=" * 60)
    
    # Extract the file path from the tool's input arguments
    file_path = input_data['tool_input'].get('file_path', '')
    
    # Block the operation if targeting a .ipynb file
    if file_path.endswith('.ipynb'):
        print("DECISION: BLOCK ACCESS TO NOTEBOOK FILE")
        print("PreToolUse: DENIED")
        print("=" * 60 + "\n")
        return {
            'hookSpecificOutput': {
                'hookEventName': input_data['hook_event_name'],
                'permissionDecision': 'deny',
                'permissionDecisionReason': 'Jupyter notebook files are protected and cannot be accessed'
            }
        }
    
    print("DECISION: ALLOW FILE ACCESS")
    print("PreToolUse: ALLOWED")
    print("=" * 60 + "\n")
    return {}

# Test with Read tool
async def test_notebook_protection():
    print("\n" + "=" * 60)
    print("Testing notebook file protection")
    print("=" * 60)
    
    # Configure hooks in options
    options = ClaudeAgentOptions(
        hooks={
            # Apply to Read, Write, and Edit tools using regex
            'PreToolUse': [HookMatcher(matcher='Read|Write|Edit', hooks=[protect_notebook_files])]
        },
        permission_mode="bypassPermissions"
    )
    
    # Use ClaudeSDKClient for hooks
    async with ClaudeSDKClient(options) as client:
        # Send message
        async def send_message():
            yield {
                "type": "user",
                "message": {
                    "role": "user",
                    "content": "Read the file 01_session_management.ipynb"
                }
            }
        
        await client.query(send_message())
        
        # Receive and display response
        async for message in client.receive_response():
            print_message(message)
            if type(message).__name__ == "ResultMessage":
                break

await test_notebook_protection()


Testing notebook file protection

Assistant: I'll find and read the notebook file.

Tool Call: Bash
  Description: Locate the notebook file
  Arguments: {
      "command": "find /Users/sajal/projects/teaching/oreilly/claude_agent_sdk_course -name \"01_session_management.ipynb\" 2>/dev/null"
}

Tool Result: /Users/sajal/projects/teaching/oreilly/claude_agent_sdk_course/claude_agent_sdk_course_code/code/examples/module_2/01_session_management.ipynb

Tool Call: Read
  Arguments: {
      "file_path": "/Users/sajal/projects/teaching/oreilly/claude_agent_sdk_course/claude_agent_sdk_course_code/code/examples/module_2/01_session_management.ipynb"
}

HOOK FIRED: PreToolUse
Tool Name: Read
Tool Use ID: toolu_01WNTEeJcEjFFzAR6b6czp4W
Tool Input: {
  "file_path": "/Users/sajal/projects/teaching/oreilly/claude_agent_sdk_course/claude_agent_sdk_course_code/code/examples/module_2/01_session_management.ipynb"
}
DECISION: BLOCK ACCESS TO NOTEBOOK FILE
PreToolUse: DENIED


Tool Error: Jupyter notebook 

### HookMatcher Configuration

| Option | Type | Description |
|--------|------|-------------|
| `matcher` | `string` | Regex pattern to match tool names (e.g., `'Bash'`, `'Write|Edit'`, `'^mcp__'`) |
| `hooks` | `list` | Array of callback functions to execute when the pattern matches |
| `timeout` | `number` | Timeout in seconds (default: 60); increase for hooks that make API calls |